In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import random
import math

import pandas as pd

import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots
import fly_plot_lib.plot as fpl

import scipy.stats
from scipy.stats import gaussian_kde

In [ ]:
from splitflow import unifying_algo_analysis_helper as uaah
from splitflow.set_zorder_functions import *

In [ ]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

In [ ]:
MARKERSIZE = 0.75
MODEL_LINE_COLOR = 'lightcoral'
TRANSLATION = True

# Variable wind colors (Figure 5): all speeds light purple, 20 cm/s highlighted. Gray is reserved for sham data.
VARWIND_COLOR = '#c4aee0ff'
LOW_WIND_HIGHLIGHT_COLOR = '#a245ffff'  # 20 cm/s; same purple as the 'Low wind (20 cm/s)' label in panel B
MIX_LINE_COLOR = '#9a82baff'  # darker purple outline for the 0 & 60 cm/s mixture curve (matches its label)

# the mixture/null/bootstrap analysis is random; fix the seed so reruns reproduce the figure
np.random.seed(0)

In [ ]:
FIGURE_NAME = 'unifying_analysis.svg'

In [ ]:
if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

In [ ]:
def get_median_slope_and_axis_ratio(fname):
    df = pd.read_parquet(fname)

    slopes = []
    axis_ratios = []
    rotation = []

    for obj_id in df.obj_id_unique_event.unique():
        data = df[df.obj_id_unique_event==obj_id].copy()
        #slopes.append(data.slope.abs().median())

        data['slope_abs'] = data.slope.abs()
        slopes.append( uaa.get_weighted_value(data, col='slope_abs', use='rmse_affine') )

        axis_ratios.append( uaa.get_weighted_value(data, col='axis_ratio', use='rmse_affine') )

        data.rotation = data.rotation.abs()
        rotation.append( uaa.get_weighted_value(data, col='rotation', use='rmse_affine') )

            
    results = {'slopes': np.array(slopes)/0.01,
               'axis_ratios': np.array(axis_ratios),
               'rotation': np.array(rotation),
               #'percent_affine_improvements': np.array(percent_affine_improvements),
              }
    
    return results

In [ ]:
df = pd.read_parquet('../../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet')

In [ ]:
df.fan_speed_percent.unique()

In [ ]:
# metadata = {'../unifying_algo_results/varwind_N200_12.0_translation' + str(TRANSLATION) + '.parquet': [1, 'gray', 'low', 0], 
#             '../unifying_algo_results/varwind_N200_20.0_translation' + str(TRANSLATION) + '.parquet': [2, 'gray', 'low', 0.03],
#             '../unifying_algo_results/varwind_N200_23.0_translation' + str(TRANSLATION) + '.parquet': [3, 'gray', 'low', 0.1],
#             '../unifying_algo_results/varwind_N200_28.0_translation' + str(TRANSLATION) + '.parquet': [4, '#bd7bffff', 'low', 0.2],
#             '../unifying_algo_results/varwind_N200_40.0_translation' + str(TRANSLATION) + '.parquet': [5, 'gray', 'low', 0.3],
#             '../unifying_algo_results/varwind_N200_90.0_translation' + str(TRANSLATION) + '.parquet': [6, 'gray', 'low', 0.6],
#            }

In [ ]:
# metadata = {'../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-12-fan_0-cms_all-traj_align_FLASH' + '.parquet': [1, 'gray', 'low', 0], 
#             '../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-20-fan_0-03-cms_all-traj_align_FLASH' + '.parquet': [2, 'gray', 'low', 0.03],
#             '../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-23-fan_0-1-cms_all-traj_align_FLASH' + '.parquet': [3, '#bd7bffff', 'low', 0.1],
#             '../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-28-fan_0-2-cms_all-traj_align_FLASH' + '.parquet': [4, 'gray', 'low', 0.2],
#             '../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-40-fan_0-3-cms_all-traj_align_FLASH' + '.parquet': [5, 'gray', 'low', 0.3],
#             '../unifying_algo_results/var_wind_kevin_analyzed/' + 'var-wind-90-fan_0-6-cms_all-traj_align_FLASH' + '.parquet': [6, 'gray', 'low', 0.6],
#            }

In [ ]:
from splitflow.unifying_results import result_path
metadata = {str(result_path('' + 'varwind_N700_flash_12.0_translationTrue' + '.parquet')): [1, VARWIND_COLOR, 'low', 0], 
            str(result_path('' + 'varwind_N700_flash_20.0_translationTrue' + '.parquet')): [2, VARWIND_COLOR, 'low', 0.03],
            str(result_path('' + 'varwind_N700_flash_23.0_translationTrue' + '.parquet')): [3, VARWIND_COLOR, 'low', 0.1],
            str(result_path('' + 'varwind_N700_flash_28.0_translationTrue' + '.parquet')): [4, LOW_WIND_HIGHLIGHT_COLOR, 'low', 0.2],
            str(result_path('' + 'varwind_N700_flash_40.0_translationTrue' + '.parquet')): [5, VARWIND_COLOR, 'low', 0.3],
            str(result_path('' + 'varwind_N700_flash_90.0_translationTrue' + '.parquet')): [6, VARWIND_COLOR, 'low', 0.6],
           }

In [ ]:
fname = list(metadata.keys())[0]
low_12_axis_ratio = get_median_slope_and_axis_ratio(fname)['axis_ratios']

fname = list(metadata.keys())[3]
low_wind_df = get_median_slope_and_axis_ratio(fname)['axis_ratios']

fname = list(metadata.keys())[5]
low_90_axis_ratio = get_median_slope_and_axis_ratio(fname)['axis_ratios']

In [ ]:
len(low_90_axis_ratio)

In [ ]:
def get_mix(mix_factor, n_total, shuffle=True):

    m1 = int(mix_factor*n_total)
    m2 = int( (1-mix_factor)*n_total)

    arr1 = np.random.choice(low_12_axis_ratio, m1)
    arr2 = np.random.choice(low_90_axis_ratio, m2)

    if mix_factor == 0:
        return arr2
    elif mix_factor == 1:
        return arr1
    else:
        mix = np.hstack([arr1, arr2])
        
    return mix

### Find closest mix

In [ ]:
best_mix = None
best_stat = 100
best_pval = 1
best_mix_factor = None
for mix_factor in np.linspace(0, 1, 10): #[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        mix = get_mix(mix_factor, len(low_wind_df))
        anderson_result = scipy.stats.anderson_ksamp([low_wind_df, mix])
        ks_result = scipy.stats.ks_2samp(low_wind_df, mix)
        
        stat_result = ks_result
        print(mix_factor, stat_result.statistic)
        if stat_result.statistic < best_stat:
            best_stat = stat_result.statistic
            best_pval = stat_result.pvalue
            best_mix = copy.copy(mix)
            best_mix_factor = mix_factor
print('best mix: ' + str(best_mix_factor))

In [ ]:
len(np.linspace(0, 1, 10))

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

if 0:
    kde = gaussian_kde(low_12_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
    x = np.linspace(low_12_axis_ratio.min(), low_12_axis_ratio.max(), 300)
    ax.plot(x, kde(x), color='gray')
    ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading
    
    kde = gaussian_kde(low_90_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
    x = np.linspace(low_90_axis_ratio.min(), low_90_axis_ratio.max(), 300)
    ax.plot(x, kde(x), color='gray')
    ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading
else:
    kde = gaussian_kde(best_mix, bw_method=0.3)  # lower = tighter, higher = smoother
    x = np.linspace(best_mix.min(), best_mix.max(), 300)
    ax.plot(x, kde(x), color='gray')
    ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading

kde = gaussian_kde(low_wind_df, bw_method=0.3)  # lower = tighter, higher = smoother
x = np.linspace(low_wind_df.min(), low_wind_df.max(), 300)
ax.plot(x, kde(x), color='#a245ffff')
ax.fill_between(x, kde(x), alpha=0.3, color='#a245ffff')  # optional shading

# fifi plot

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
if 0:
    fig = plt.figure()
    ax = fig.add_subplot(111)
else:
    ax = layout.axes[('comparison', 'comparison')]
    
# kde = gaussian_kde(low_12_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
# x = np.linspace(low_12_axis_ratio.min(), low_12_axis_ratio.max(), 300)
# ax.plot(x, kde(x), color='gray')
# ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading

# kde = gaussian_kde(low_90_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
# x = np.linspace(low_90_axis_ratio.min(), low_90_axis_ratio.max(), 300)
# ax.plot(x, kde(x), color='gray')
# ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading

# kde = gaussian_kde(low_wind_df, bw_method=0.3)  # lower = tighter, higher = smoother
# x = np.linspace(low_wind_df.min(), low_wind_df.max(), 300)
# ax.plot(x, kde(x), color='#a245ffff')
# ax.fill_between(x, kde(x), alpha=0.3, color='#a245ffff')  # optional shading

if 0:
    kde = gaussian_kde(low_12_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
    x = np.linspace(low_12_axis_ratio.min(), low_12_axis_ratio.max(), 300)
    ax.plot(x, kde(x), color='gray')
    ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading
    
    kde = gaussian_kde(low_90_axis_ratio, bw_method=0.3)  # lower = tighter, higher = smoother
    x = np.linspace(low_90_axis_ratio.min(), low_90_axis_ratio.max(), 300)
    ax.plot(x, kde(x), color='gray')
    ax.fill_between(x, kde(x), alpha=0.3, color='gray')  # optional shading
else:
    # Expected density of the best 0 & 60 cm/s mixture, instead of the KDE of one random mixture draw
    # (whose shape varies a lot from draw to draw). The mix fraction is chosen without resampling: the
    # same candidate fractions as the test above, scored by the KS distance between the 20 cm/s data and
    # the mixture CDF f*F_0 + (1-f)*F_60.
    grid = np.sort(np.hstack([low_12_axis_ratio, low_90_axis_ratio, low_wind_df]))
    ecdf = lambda a: np.searchsorted(np.sort(a), grid, side='right') / len(a)
    F_12, F_90, F_low = ecdf(low_12_axis_ratio), ecdf(low_90_axis_ratio), ecdf(low_wind_df)
    mix_factors = np.linspace(0, 1, 10)
    ks_expected = [np.max(np.abs(F_low - (f*F_12 + (1-f)*F_90))) for f in mix_factors]
    plot_mix_factor = mix_factors[np.argmin(ks_expected)]
    print('mix factor for plot (fraction 0 cm/s): %.2f, KS = %.3f' % (plot_mix_factor, np.min(ks_expected)))

    x = np.linspace(0, 1, 300)
    mix_density = (plot_mix_factor * gaussian_kde(low_12_axis_ratio, bw_method=0.3)(x)
                   + (1 - plot_mix_factor) * gaussian_kde(low_90_axis_ratio, bw_method=0.3)(x))
    ax.fill_between(x, mix_density, alpha=0.3, color=VARWIND_COLOR, linewidth=0)
    ax.plot(x, mix_density, color=MIX_LINE_COLOR)

kde = gaussian_kde(low_wind_df, bw_method=0.3)  # lower = tighter, higher = smoother
x = np.linspace(low_wind_df.min(), low_wind_df.max(), 300)
ax.plot(x, kde(x), color=LOW_WIND_HIGHLIGHT_COLOR)
ax.fill_between(x, kde(x), alpha=0.3, color=LOW_WIND_HIGHLIGHT_COLOR)  # optional shading

ax.set_xlim(0, 1)
ax.set_ylim(0, 3)

ax.set_ylabel('Density', labelpad=-2)
ax.set_xlabel('Axis ratio', labelpad=1)

ax.set_xticks([0, .5, 1])
ax.set_xticklabels(['0', '0.5', '1'])

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['0', '', '', '3'])

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)

layout.append_figure_to_layer(layout.figures['comparison'], 'comparison', cleartarget=True)
layout.write_svg(FIGURE_NAME)

In [ ]:
# if 1:
#     fig = plt.figure()
#     ax = fig.add_subplot(111)
# else:
#     ax = layout.axes[('comparison', 'comparison')]

# ax.hist(low_wind_df, color='#a245ffff', bins=15, density=True)
# #ax.hist(low_12_axis_ratio, color='gray', density=True, alpha=0.3)
# #ax.hist(low_90_axis_ratio, color='gray', density=True, alpha=0.3)

# for mix_factor in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
#     mix = get_mix(mix_factor, len(low_wind_df))
#     ax.hist(mix, color='gray', 
#             density=True, 
#             bins=10,
#             alpha=0.3)
#     if mix_factor < 0.1 or mix_factor > 0.9:
#         ax.hist(mix, 
#                 color='black',
#                 histtype='step',
#                 linewidth=0.25,
#                 density=True, 
#                 bins=10,
#                 alpha=1,
#                 zorder=1000)
    

#     anderson_result = scipy.stats.anderson_ksamp([low_wind_df, mix])
#     ks_result = scipy.stats.ks_2samp(low_wind_df, mix)
#     print('pvals: ', ks_result.pvalue, anderson_result.pvalue)

# #ax.hist(low_90_axis_ratio, histtype='step', bins=15, color='black', density=True, linewidth=0.25)
# #ax.hist(low_12_axis_ratio, histtype='step', bins=15, color='black', density=True, linewidth=0.25)
# ax.hist(low_wind_df, histtype='step', bins=15, color='purple', density=True)

# Null distributions

In [ ]:
max_statistic = [] 
corresponding_pval = []

for i in range(5000):
    test_mix = get_mix(np.random.uniform(0,1), len(low_wind_df))

    # Find the smallest test statistic / largest pval -- i.e. the mixture that is most similar to the distribution
    best_stat = 100
    for mix_factor in np.linspace(0, 1, 10): #[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        mix = get_mix(mix_factor, len(low_wind_df))
        anderson_result = scipy.stats.anderson_ksamp([test_mix, mix])
        ks_result = scipy.stats.ks_2samp(test_mix, mix)
        
        stat_result = ks_result
        if stat_result.statistic < best_stat:
            best_stat = stat_result.statistic
            best_pval = stat_result.pvalue
    max_statistic.append(best_stat)
    corresponding_pval.append(best_pval)

# Bootstrap observed pvals

In [ ]:
observed_statistic_distribution = []
observed_pval_distribution = []

for i in range(1000):
    observed_statistic = 100
    observed_pval = None
    for mix_factor in np.linspace(0, 1, 10): #[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        mix = get_mix(mix_factor, len(low_wind_df))
        anderson_result = scipy.stats.anderson_ksamp([low_wind_df, mix])
        ks_result = scipy.stats.ks_2samp(low_wind_df, mix)

        stat_result = ks_result
        if stat_result.statistic < observed_statistic:
            observed_statistic = stat_result.statistic
            observed_pval = stat_result.pvalue
    
    
    observed_statistic_distribution.append(observed_statistic)
    observed_pval_distribution.append(observed_pval)

In [ ]:
observed_statistic_distribution = np.sort(observed_statistic_distribution)
conf_low = observed_statistic_distribution[int(0.025*len(observed_statistic_distribution))]
conf_high = observed_statistic_distribution[int(0.975*len(observed_statistic_distribution))]

In [ ]:
observed_pval_distribution = np.sort(observed_pval_distribution)
conf_low_pval = observed_pval_distribution[int(0.025*len(observed_pval_distribution))]
conf_high_pval = observed_pval_distribution[int(0.975*len(observed_pval_distribution))]

# Make plots

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
if 0:
    fig = plt.figure()
    ax = fig.add_subplot(111)
else:
    ax = layout.axes[('bootstrap', 'stat')]
    
ax.hist(max_statistic, density=True, color=VARWIND_COLOR)
#plt.vlines(observed_statistic, 0, 1, color='red')
#ax.hist(observed_statistic_distribution, density=True)
ax.vlines(np.mean(observed_statistic_distribution), 0, 30, color=LOW_WIND_HIGHLIGHT_COLOR)
#ax.fill_betweenx([0,1], conf_low, conf_high, color='purple', alpha=0.3)

# keep the observed statistic (purple line) inside the axis
ax.set_xlim(0.02, 0.125)
assert np.mean(observed_statistic_distribution) < 0.125, 'observed K-S statistic is off the axis'
ax.set_ylim(0, 30)

ax.set_ylabel('Density', labelpad=-5)
ax.set_xlabel('K-S stat', labelpad=1)

ax.set_xticks([0.02, 0.12])  # the axis line extends past the observed statistic
#ax.set_xticklabels([0, .2, .4, .6, .8, 1])

ax.set_yticks([0, 30])

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)

In [ ]:
if 0:
    fig = plt.figure()
    ax = fig.add_subplot(111)
else:
    ax = layout.axes[('bootstrap', 'pval')]

bins = np.logspace(np.log10(np.min(corresponding_pval)), np.log10(np.max(corresponding_pval)), 10)

ax.hist(corresponding_pval, bins=bins, density=True, color=VARWIND_COLOR)
ax.vlines(np.mean(observed_pval_distribution), 0, 2, color=LOW_WIND_HIGHLIGHT_COLOR)
#ax.fill_betweenx([0,1], conf_low_pval, conf_high_pval, color='purple', alpha=0.3)
ax.set_xscale('log')


ax.set_xlim(-2, 0)
ax.set_xticks([1e-2, 1e-1, 1e0])
ax.set_ylim(0, 2)

ax.set_ylabel('Density', labelpad=-1)
ax.set_xlabel('K-S pval', labelpad=2)

ax.set_yticks([0, 2])

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)
ax.minorticks_off()

In [ ]:
item = layout.svgitems['text_ks_pval']
# four decimal places (p is of order 1e-4)
item.text = 'p=%.4f' % np.mean(observed_pval_distribution)
layout.apply_svg_attrs()

In [ ]:
np.mean(observed_pval_distribution)

In [ ]:
layout.append_figure_to_layer(layout.figures['bootstrap'], 'bootstrap', cleartarget=True)
layout.write_svg(FIGURE_NAME)